# T05: Count input and output tokens

**Owner:** Alex · **Task:** T05 in the project plan

This notebook adds two columns to `data/responses_clean.csv` (the output of T04):

- **`input_tokens`**: tokens in the `prompt` sent to the model.
- **`output_tokens`**: tokens in the model's `response`.

**Why only the prompt?** In RAGTruth the prompt already contains the source text (the question passages, the business data, or the news article). Adding `source_info` on top would count the source twice.

**Tokenizer:** OpenAI's `cl100k_base` encoding (via `tiktoken`). Counts are exact for GPT-3.5 and GPT-4 and approximate for Llama-2 and Mistral, which use their own tokenizers. These counts feed the cost analysis in T14, T17, and T19.

Run the cells top to bottom. The notebook can be rerun safely: it recomputes the token columns each time.

## 1. Setup
Install the packages once if needed, then import them.

In [1]:
# Uncomment the next line the first time you run this notebook.
# %pip install tiktoken pandas

from pathlib import Path

import pandas as pd
import tiktoken

## 2. File locations
The notebook works whether you start Jupyter from the repository root or from `notebooks/`.

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data" / "responses_clean.csv"
TOKENS_PATH = ROOT / "data" / "processed" / "T05_token_counts.csv"

print("Clean file:", CLEAN_PATH.relative_to(ROOT), "| exists:", CLEAN_PATH.exists())

Clean file: data/responses_clean.csv | exists: True


## 3. Load the clean data from T04

In [3]:
df = pd.read_csv(CLEAN_PATH)
print(f"{len(df):,} responses")
df[["id", "model", "task_type"]].head()

17,617 responses


,id,model,task_type
0,0,gpt-4-0613,Summary
1,1,gpt-3.5-turbo-0613,Summary
2,2,mistral-7B-instruct,Summary
3,3,llama-2-7b-chat,Summary
4,4,llama-2-13b-chat,Summary


## 4. Load the tokenizer and define the counting function
`disallowed_special=()` treats strings such as `<|endoftext|>` as ordinary text instead of raising an error. Missing text counts as 0 tokens.

In [4]:
encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    """Return the number of cl100k_base tokens in text (0 if missing)."""
    if pd.isna(text):
        return 0
    return len(encoding.encode(str(text), disallowed_special=()))


print("Check:", encoding.encode("hello world"))  # expected [15339, 1917]

Check: [15339, 1917]


## 5. Count tokens for every response
This takes about a minute.

In [5]:
df["input_tokens"] = df["prompt"].apply(count_tokens)
df["output_tokens"] = df["response"].apply(count_tokens)

df[["input_tokens", "output_tokens"]].describe().round(1)

,input_tokens,output_tokens
count,17617.0,17617.0
mean,615.7,164.2
std,310.2,70.0
min,155.0,1.0
25%,402.0,114.0
50%,552.0,156.0
75%,753.0,204.0
max,2233.0,774.0


## 6. Sanity checks
- No response should have zero tokens.
- Input tokens should be about the same for every model, because all six models received the same prompts.

In [6]:
print("Zero input tokens: ", int((df["input_tokens"] == 0).sum()))
print("Zero output tokens:", int((df["output_tokens"] == 0).sum()))

df.groupby("model")[["input_tokens", "output_tokens"]].mean().round(1)

Zero input tokens:  0
Zero output tokens: 0


,input_tokens,output_tokens
model,,
gpt-3.5-turbo-0613,618.6,181.1
gpt-4-0613,616.4,143.0
llama-2-13b-chat,613.6,175.6
llama-2-70b-chat,613.9,165.0
llama-2-7b-chat,614.4,173.1
mistral-7B-instruct,617.2,147.1


In [7]:
df.groupby("task_type")[["input_tokens", "output_tokens"]].mean().round(1)

,input_tokens,output_tokens
task_type,,
Data2txt,750.9,199.8
QA,374.0,151.6
Summary,714.0,137.9


## 7. Save the results
- The token columns are written back into `data/responses_clean.csv` (local only: the file is about 104 MB, over GitHub's 100 MB limit).
- A small shareable file, `data/processed/T05_token_counts.csv`, holds just `id`, `model`, and the two counts. It is small enough to commit, and it joins to `data/processed/features.csv` on `id`.

In [8]:
df.to_csv(CLEAN_PATH, index=False)

TOKENS_PATH.parent.mkdir(parents=True, exist_ok=True)
df[["id", "model", "input_tokens", "output_tokens"]].to_csv(TOKENS_PATH, index=False)

print("Updated:", CLEAN_PATH.relative_to(ROOT))
print("Wrote:  ", TOKENS_PATH.relative_to(ROOT), f"({TOKENS_PATH.stat().st_size / 1e6:.2f} MB)")

Updated: data/responses_clean.csv
Wrote:   data/processed/T05_token_counts.csv (0.53 MB)
